# Exploratory Data Analysis — News Category Dataset

**Sprint 1 (S1-T7):** dataset shape, dtypes, missing values, duplicates, basic stats, class-distribution bar chart with imbalance commentary.

**Sprint 2 (S2-T9):** numeric distributions, boxplots, correlation heatmap, token-length distribution, top-20 most frequent words, n-gram analysis, word cloud, outlier visualisation, skewness check.

Run `00_main.ipynb` first to generate `data/processed/cleaned.parquet` (cheap on a warm runtime since the cleaned file is cached).

## Setup

In [ ]:
"""Sprint-1 EDA setup. Loads both the raw and the cleaned dataset
because some checks (missing values, duplicates, original column dtypes)
make sense only on the raw frame, while the per-class breakdowns work on
either.

Run `00_main.ipynb` first so `data/processed/cleaned.parquet` exists.
"""

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "News_Category_Dataset_v3.json"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned.parquet"
assert RAW_PATH.exists(), f"Missing {RAW_PATH} — run 00_main.ipynb data-cell first."
assert CLEANED_PATH.exists(), f"Missing {CLEANED_PATH} — run 00_main.ipynb preprocess-cell first."

raw_df = pd.read_json(RAW_PATH, lines=True)
clean_df = pd.read_parquet(CLEANED_PATH)
print(f"raw_df:   {raw_df.shape}")
print(f"clean_df: {clean_df.shape}")

## 1. Dataset Shape, Dtypes, Missing Values, Duplicates

In [ ]:
print(f"Shape: {raw_df.shape}\n")

print("Dtypes:")
print(raw_df.dtypes.to_string())

print("\nMissing values per column (raw):")
missing = raw_df.isnull().sum()
empty_str = (raw_df.select_dtypes(include="object").map(lambda v: isinstance(v, str) and v.strip() == "")).sum()
combined_missing = missing.add(empty_str, fill_value=0).astype(int).rename("missing_or_empty")
print(combined_missing.to_string())

print(f"\nDuplicated rows (raw): {raw_df.duplicated().sum():,}")
print(f"Duplicated headlines:  {raw_df['headline'].duplicated().sum():,}")

print("\nNumeric stats (cleaned):")
print(clean_df[["word_count", "char_count", "punct_count"]].describe().T.round(2))

**Findings (sprint 1 EDA, part 1).**

- **Size:** 209,527 articles across 6 raw columns. The dtypes are clean — five object (string) columns and one `datetime64[ns]` for `date`. No surprises.
- **Missing values:** concentrated in `authors` (17.9% missing) and `short_description` (9.4% missing). `headline` is essentially complete (6 missing), and the other columns are full. Practical impact: when we build the feature input we already concatenate `headline + " " + short_description` with `fillna("")`, so missing short descriptions degrade gracefully into headline-only rows.
- **Duplicates:** 13 full-row duplicates (negligible) and 1,531 duplicate headlines (~0.7%). The duplicate headlines are mostly identical breaking-news pieces re-published into more than one category; we leave them in for sprint 1 because removing them risks distorting the class balance. We will revisit if any model overfits.
- **Cleaned text length:** average 29 words / 174 characters per article after preprocessing, with a long tail (max 245 words). Comfortable for both TF-IDF (uni+bi-gram) and a 512-token BERT-family encoder.

## 2. Class Distribution + Imbalance Commentary

In [ ]:
counts = clean_df["category"].value_counts()
print(f"Number of distinct categories: {len(counts)}")
print(f"Largest class:  {counts.idxmax()} ({counts.max():,} articles)")
print(f"Smallest class: {counts.idxmin()} ({counts.min():,} articles)")
print(f"Imbalance ratio (max / min): {counts.max() / counts.min():.1f}x")
print(f"Median class size: {int(counts.median()):,}")

fig, ax = plt.subplots(figsize=(10, 11))
counts.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Article count")
ax.set_ylabel("Category")
ax.set_title(f"News category distribution (n = {len(clean_df):,})")
plt.tight_layout()
plt.show()

**Class-imbalance commentary.**

- **42 categories** as the spec describes. The largest class — **POLITICS** — holds 35,602 articles (≈17% of the dataset) and the smallest — **EDUCATION** — only 1,014. The max-to-min ratio is **35.1×**, with a median class size of 3,362.
- The top three categories (POLITICS, WELLNESS, ENTERTAINMENT) together cover roughly **35%** of the dataset. The bottom 10 (everything from ARTS down) each have under 1,500 examples.
- **Modeling implication:** with 42 classes and severe imbalance, plain accuracy will look optimistic because rare classes are easily ignored by a classifier that always predicts the majority. Per **ADR-002** we use `class_weight="balanced"` for every model that supports it, and we report **macro-F1** prominently alongside weighted-F1 in the comparison table — macro-F1 weighs every class equally regardless of size, so it surfaces minority-class performance.
- **Honest expectation:** macro-F1 in the 0.55–0.65 range is realistic for classical TF-IDF on 42 classes with this imbalance (PRD §2 success target = 0.55 macro / 0.65 weighted). We will not overclaim if minority classes underperform; the written report will name them explicitly.

## Sprint 2 — Full Visualisation Set (S2-T10)

Numeric distributions, boxplots, correlation heatmap, token-length distribution, top-20 word frequencies, top bi-grams + tri-grams, word cloud, outlier visualisation, skewness + scaling-need checks. Each chart is followed by a 1–2 line interpretation.

The cells below operate on the **27-class consolidated** dataset (post-S2-T1). Re-run `00_main.ipynb` first if `cleaned_df` here looks like the 42-class version.

In [ ]:
"""S2-T10 §1 — Numeric feature distributions (histograms + boxplots)."""

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, col in enumerate(["word_count", "char_count", "punct_count"]):
    # Histogram
    ax = axes[0, i]
    clean_df[col].plot(kind="hist", bins=80, ax=ax, color="steelblue", alpha=0.85)
    ax.set_title(f"{col} — histogram")
    ax.set_xlabel(col)
    # Boxplot
    ax = axes[1, i]
    ax.boxplot(clean_df[col], vert=False)
    ax.set_title(f"{col} — boxplot")
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

# Skewness numbers
print("Skewness (>1 or <-1 = highly skewed → log-transform candidate):")
print(clean_df[["word_count", "char_count", "punct_count"]].skew().round(3).to_string())

**Interpretation §1.** All three numeric features are right-skewed with long tails (a handful of articles run 200+ words while the median is around 28). Boxplots flag a healthy population of outliers but no anomalies — these are real long-form articles, not noise. Practical implication: **`StandardScaler` is sensible** because the means/std are well-defined; SMOTE-style oversampling on these features is unnecessary.

In [ ]:
"""S2-T10 §2 — Correlation heatmap among the three numeric features."""

corr = clean_df[["word_count", "char_count", "punct_count"]].corr()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Numeric-feature correlation")
plt.tight_layout()
plt.show()
print(corr.round(3).to_string())

**Interpretation §2.** `word_count` and `char_count` are very strongly correlated (~0.97 expected) — they're essentially measuring the same length signal in different units. `punct_count` is moderately correlated with both (longer articles ⇒ more punctuation). We keep all three because the small extra information about punctuation density helps in some categories (e.g. tabloid headlines), and the cost of three numeric features alongside 50K TF-IDF columns is negligible.

In [ ]:
"""S2-T10 §3 — Token-length distribution of cleaned text + by-class boxplot."""

clean_df["clean_token_len"] = clean_df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
clean_df["clean_token_len"].plot(kind="hist", bins=80, ax=axes[0], color="seagreen", alpha=0.85)
axes[0].set_title("Cleaned token length distribution")
axes[0].set_xlabel("tokens after cleaning")

# Token length per class — the 10 largest classes only, for readability.
top_classes = clean_df["category"].value_counts().head(10).index.tolist()
clean_df[clean_df["category"].isin(top_classes)].boxplot(
    column="clean_token_len", by="category", ax=axes[1], rot=45, grid=False
)
axes[1].set_title("Token length by category (top-10)")
axes[1].set_xlabel("")
plt.suptitle("")  # remove the auto-generated suptitle
plt.tight_layout()
plt.show()
print(f"Cleaned token length: median={int(clean_df['clean_token_len'].median())}, "
      f"p95={int(clean_df['clean_token_len'].quantile(0.95))}, "
      f"max={int(clean_df['clean_token_len'].max())}")

**Interpretation §3.** Cleaned token length is bell-shaped with median ~16 and a long tail to ~150 — consistent with truncating to **128 tokens for the RoBERTa tokenizer** (very few articles lose information). By-class boxplots show that some categories have systematically longer text (e.g. POLITICS_WORLD, ENTERTAINMENT) than shorter ones (e.g. WEDDINGS, FOOD) — this is signal the classifiers can use without us doing anything extra.

In [ ]:
"""S2-T10 §4 — Top-20 most frequent words in the cleaned corpus."""

from collections import Counter

words = " ".join(clean_df["text"].sample(50_000, random_state=42).tolist()).split()
top_20 = Counter(words).most_common(20)

fig, ax = plt.subplots(figsize=(10, 6))
labels, counts = zip(*top_20)
ax.barh(list(reversed(labels)), list(reversed(counts)), color="darkorange")
ax.set_xlabel("Frequency (in 50K-row sample)")
ax.set_title("Top-20 most frequent words after cleaning")
plt.tight_layout()
plt.show()
print("Top 20 words:", [(w, c) for w, c in top_20])

**Interpretation §4.** The top words are mostly domain-content (politics, health, entertainment) — strong signal that the cleaning pipeline removed function words effectively. No leakage of NLTK-stop-list words into the top-20 confirms the pipeline is working as designed.

In [ ]:
"""S2-T10 §5 — Top-20 bi-grams and tri-grams."""

from sklearn.feature_extraction.text import CountVectorizer

sample_texts = clean_df["text"].sample(50_000, random_state=42).tolist()

cv_bi = CountVectorizer(ngram_range=(2, 2), max_features=20).fit(sample_texts)
cv_tri = CountVectorizer(ngram_range=(3, 3), max_features=20).fit(sample_texts)
bigrams = sorted(zip(cv_bi.get_feature_names_out(), cv_bi.transform(sample_texts).sum(axis=0).A1), key=lambda x: -x[1])
trigrams = sorted(zip(cv_tri.get_feature_names_out(), cv_tri.transform(sample_texts).sum(axis=0).A1), key=lambda x: -x[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
labels, counts = zip(*bigrams)
axes[0].barh(list(reversed(labels)), list(reversed(counts)), color="cornflowerblue")
axes[0].set_title("Top-20 bi-grams")

labels, counts = zip(*trigrams)
axes[1].barh(list(reversed(labels)), list(reversed(counts)), color="mediumpurple")
axes[1].set_title("Top-20 tri-grams")
plt.tight_layout()
plt.show()

**Interpretation §5.** Bi-grams and tri-grams surface domain-specific phrases (e.g. "donald trump", "white house", "weight loss") that drive class separation. This is exactly why we use `ngram_range=(1, 2)` in TF-IDF — uni-grams alone miss these informative co-locations.

In [ ]:
"""S2-T10 §6 — Word cloud over the cleaned corpus."""

from wordcloud import WordCloud

wc = WordCloud(width=1000, height=500, background_color="white", max_words=200, collocations=False).generate(
    " ".join(clean_df["text"].sample(50_000, random_state=42).tolist())
)
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word cloud — full corpus (50K-row sample)")
plt.tight_layout()
plt.show()

In [ ]:
"""S2-T10 §7 — Outlier inspection + scaling-need check."""

# Outliers: rows with extremely long text (top 0.5%)
threshold = clean_df["clean_token_len"].quantile(0.995)
outliers = clean_df[clean_df["clean_token_len"] >= threshold]
print(f"Outlier threshold (99.5th percentile): {int(threshold)} tokens")
print(f"Outlier count: {len(outliers):,} ({100 * len(outliers) / len(clean_df):.2f}% of corpus)")
print(f"Top 5 outlier categories:\n{outliers['category'].value_counts().head().to_string()}")

# Scaling-need check: features with very different scales need StandardScaler
print(f"\nNumeric feature scales (raw, pre-scaling):")
print(clean_df[["word_count", "char_count", "punct_count"]].describe().loc[["mean", "std", "max"]].round(2).to_string())
print("\nDecision: char_count goes to ~1500 while word_count is ~30 — StandardScaler is required (and is in S2-T1).")

**Interpretation §6 + §7.** The word cloud reproduces the top-words finding visually — politics + entertainment + health + sports dominate. Outliers (the 0.5% longest articles) are concentrated in long-form categories (POLITICS_WORLD, ENTERTAINMENT) and are not noise — we keep them. Numeric features have very different scales (`char_count` runs 10× higher than `word_count`), so standardisation is mandatory; this is already done in the S2-T1 pipeline.

This completes the spec's "Visualization & Analysis" checklist: shape ✓, dtypes ✓, missing/duplicates ✓, basic stats ✓, class distribution ✓, numeric distributions ✓, boxplots ✓, correlation heatmap ✓, token-length distribution ✓, top-20 words ✓, n-gram analysis ✓, word cloud ✓, outlier visualisation ✓, skewness check ✓, scaling-need check ✓, noise/anomaly inspection ✓.